# Sprawdzenie dostępności bibliotek Delta Lake

Na początku wykonaj poniższe paragrafy tworzące tabelę Delta Lake, jeśli wszystko się powiedzie możesz przejść do sekcji ***Wprowadzenie*** pominąć następną sekcję poświęconą konfiguracji. 
W przeciwnym razie wykonaj polecenia z sekcji ***Konfiguracja***.

In [1]:
import pyspark
from delta import *
from pyspark.sql.functions import col, explode, array


builder = pyspark.sql.SparkSession.builder.appName("MyApp") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.databricks.delta.schema.autoMerge.enabled", "true")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/hadoop/.ivy2.5.2/cache
The jars for the packages stored in: /home/hadoop/.ivy2.5.2/jars
io.delta#delta-spark_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4ca93226-4b7e-4841-a77c-595ec7d26821;1.0
	confs: [default]
	found io.delta#delta-spark_2.13;4.0.0 in central
	found io.delta#delta-storage;4.0.0 in central
	found org.antlr#antlr4-runtime;4.13.1 in central
:: resolution report :: resolve 67ms :: artifacts dl 2ms
	:: modules in use:
	io.delta#delta-spark_2.13;4.0.0 from central in [default]
	io.delta#delta-storage;4.0.0 from central in [default]
	org.antlr#antlr4-runtime;4.13.1 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|

In [2]:
data = (
    spark.range(0, 5)
    .selectExpr("id as x")
    .withColumn("y", explode(array(col("x"))))
    .select("x", "y")
)

# Zapis danych do formatu Delta
data.write.format("delta").mode("overwrite").save("/tmp/delta-table2")

25/12/07 23:13:42 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

# Konfiguracja
Jeśli nie masz dostępnej biblioteki *Delta Lake*, dokonaj instalacji właściwej w Twoim przypadku wersji (ewentualny brak pyspark zostanie przy okazji także rozwiązany ze względu na występujące pomiędzy tymi bibliotekami zależności) 

In [3]:
pip install delta-spark==4.0.0

Note: you may need to restart the kernel to use updated packages.


# Wprowadzenie

## Skonfigurowanie danych źródłowych 

Zanim zaczniemy korzystać i poznawać funkcjonalności biblioteki Delta Lake skonfigurujmy tabelę z danymi źródłowymi. 
A następnie uruchom go tworząc tymczasową perspektywę na przygotowanych uprzednio danych źródłowych. 

In [3]:
source_df = (
    spark.read
    .option("header", True)
    .option("quote", "\"")
    .csv("/tmp/DeltaLakeSourceData")
)

# Wyświetlenie schematu ramki danych
source_df.printSchema()

# Utworzenie tymczasowej tabeli
source_df.createOrReplaceTempView("source_data")

root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- address: string (nullable = true)
 |-- zipcode: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- effectiveDate: string (nullable = true)



 
Nasze dane źródłowe zawierają informacje na temat klientów z kolejnych okresów czasu.

Uruchom poniższe zapytanie wydobywające wersje danych, które były aktualne na dzień `2021-01-01`.

In [4]:
df = spark.sql("""
select id, name, address, zipcode, city, country, effectiveDate
from  (
    select id, name, address, zipcode, city, country, effectiveDate, 
           rank() over (partition by id order by to_date(effectiveDate,"dd-MM-yyyy") desc) as version
    from   source_data
    where  to_date(effectiveDate,"dd-MM-yyyy") < to_date("2021-01-01","yyyy-MM-dd")
    ) tab
where version = 1""")

In [5]:
df.toPandas()

,id,name,address,zipcode,city,country,effectiveDate
0,10,Jin Terry,467-8297 Enim,35633573,Balıkesir,Nigeria,06-11-2020
1,100,Harriet Rojas,Ap #810-8710 Enim. St.,84541,Lipetsk,Canada,21-10-2020
2,11,Isabelle Stevenson,131-4245 Eleifend. Street,16142,Hà Giang,Russian Federation,25-11-2020
3,14,Leo Mcleod,467-8297 Enim,39153,Borås,Germany,16-10-2020
4,16,Kaitlin Landry,623-5682 Augue St.,351225,Libramont-Chevigny,Indonesia,29-10-2020
5,19,Alden Harper,Ap #579-2185 Sed Street,94671-72608,Châtellerault,Nigeria,06-12-2020
6,2,Brandon Christian,476-5064 Suspendisse Rd.,93-765,Broxburn,Russian Federation,28-11-2020
7,20,Kathleen Pugh,7018 Cras St.,3123,Ostrowiec Świętokrzyski,Peru,16-11-2020
8,26,Ulysses Dillard,1318 Tempor Rd.,S5J 6Z2,Tuscaloosa,United States,11-10-2020
9,28,Shaine Puckett,Ap #579-2185 Sed Street,85629,Cochrane,Poland,07-12-2020


W poniższych zadaniach możesz skorzystać zarówno z interfejsu w Scali jak i SQL. 

Wszystko zależy od Twoich preferencji.

Uwaga! W przypadku korzystania z SQL i wskazywania ścieżek, konieczne jest wykorzystywanie schematu `hdfs`.

Przykładowo `hdfs:/tmp/delta-customers`

## DDL

### Zadanie 1

Utwórz pustą tabelę *Delta Lake* o nazwie `customers`, której lokalizacją będzie `/tmp/delta-customers` (lub `hdfs:/tmp/delta-customers`). 

Kolumny tabeli muszą odpowiadać kolumnom danych źródłowych. 
Wszystkie kolumny powinny być ciągami znaków o długości do 200 znaków

Jeśli uruchamiasz ten notatnik po raz kolejny. Usuń zawartość katalogu z danymi tabeli Delta Lake wywołując poniższe polecenie

In [6]:
%%sh 
hadoop fs -rm -r /tmp/delta-customers

Deleted /tmp/delta-customers


### Rozwiązanie zadania 1

In [7]:
spark.sql("DROP TABLE IF EXISTS customers")

25/12/07 23:14:18 WARN HiveConf: HiveConf of name hive.enforce.bucketing does not exist
25/12/07 23:14:18 WARN HiveClientImpl: Detected HiveConf hive.execution.engine is 'tez' and will be reset to 'mr' to disable useless hive logic
Hive Session ID = ecf1b01a-7558-4620-89b9-b7e7084cd49d


DataFrame[]

In [8]:
spark.sql("""
CREATE TABLE customers (
id VARCHAR(200),
name VARCHAR(200), 
address VARCHAR(200), 
zipcode VARCHAR(200), 
city VARCHAR(200), 
country VARCHAR(200), 
effectiveDate VARCHAR(200)
) USING DELTA
LOCATION 'hdfs:/tmp/delta-customers';
""")

25/12/07 23:14:40 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider delta. Persisting data source table `spark_catalog`.`default`.`customers` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.


DataFrame[]

## DML

### Zadanie 2

Wprowadź do utworzonej przez Ciebie tabeli dane o naszych klientach obowiązujące na dzień `2021-01-01`.

### Rozwiązanie zadania 2

In [17]:
spark.sql("""
INSERT INTO customers (id, name, address, zipcode, city, country, effectiveDate)
SELECT id, name, address, zipcode, city, country, effectiveDate
FROM (
    SELECT id, name, address, zipcode, city, country, effectiveDate,
           RANK() OVER (PARTITION BY id ORDER BY to_date(effectiveDate,"dd-MM-yyyy") DESC) AS rank
    FROM source_data
    WHERE to_date(effectiveDate,"dd-MM-yyyy") < DATE "2021-01-01"
)
WHERE rank = 1
""")

DataFrame[]

In [16]:
spark.sql("""
SELECT * FROM customers
""")

DataFrame[id: string, name: string, address: string, zipcode: string, city: string, country: string, effectiveDate: string]

### Zadanie 3

Zmień kraj zamieszkania na wartość `Poland` u wszystkich klientów posiadających wartość id mniejszą niż 50. 

Zwróć uwagę, że obecnie id jest ciągiem znaków. Użyj wyrażenia `cast(id as int)` aby wykonać zadanie prawidłowo.

### Rozwiązanie zadania 3

In [18]:
spark.sql("""
UPDATE customers 
SET country = 'Poland' 
WHERE CAST(id AS INT) < 50
""")

25/12/07 23:19:41 WARN UpdateCommand: Could not validate number of records due to missing statistics.


DataFrame[num_affected_rows: bigint]

Na chwilę się zatrzymajmy. Utworzenie tabeli to wpis w metadanych. Wprowadzenie nowych danych to utworzenie nowych plików w katalogu tabeli. Czym było zmodyfikowanie tych danych? 

Wykonaj poniższe polecenie, aby przyglądnąć się zawartości katalogu należącego do tabeli `customers`

In [19]:
%%sh
hadoop fs -ls /tmp/delta-customers

Found 6 items
drwxr-xr-x   - hadoop supergroup          0 2025-12-07 23:19 /tmp/delta-customers/_delta_log
-rw-r--r--   2 hadoop supergroup       4879 2025-12-07 23:19 /tmp/delta-customers/part-00000-338f7990-ae02-46dc-9a3a-c81d45d6a065-c000.snappy.parquet
-rw-r--r--   2 hadoop supergroup       1134 2025-12-07 23:15 /tmp/delta-customers/part-00000-4b5124e6-244c-4b89-bb29-7ddebc939de2-c000.snappy.parquet
-rw-r--r--   2 hadoop supergroup       4642 2025-12-07 23:19 /tmp/delta-customers/part-00000-cd2f8f28-aba4-4ca7-8a7f-47ee6a30d464-c000.snappy.parquet
-rw-r--r--   2 hadoop supergroup       4642 2025-12-07 23:19 /tmp/delta-customers/part-00000-f92fd757-ef7d-470a-9f52-39a1f796c958-c000.snappy.parquet
-rw-r--r--   2 hadoop supergroup       4879 2025-12-07 23:19 /tmp/delta-customers/part-00001-279e052d-b122-4ceb-8177-4df85df2ac54-c000.snappy.parquet


Mamy katalog z logiem transakcyjnym i dwa pliki, które prawie nie różnią się wielkością.

Spróbujmy zrozumieć znaczenie tych plików zaglądając do historii zmian w naszej tabeli. 
Koniecznie przestudiuj kolumnę `operationMetrics`

In [20]:
df = spark.sql("describe history customers")

In [21]:
df.toPandas()

,version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,3,2025-12-07 23:19:41.970,None,None,UPDATE,"{'predicate': '[""(cast(id#1275 as int) < 50)""]'}",None,None,None,2.0,Serializable,False,"{'numDeletionVectorsUpdated': '0', 'numAddedFi...",None,Apache-Spark/4.0.1 Delta-Lake/4.0.0
1,2,2025-12-07 23:19:27.255,None,None,WRITE,"{'mode': 'Append', 'partitionBy': '[]'}",None,None,None,1.0,Serializable,True,"{'numOutputRows': '32', 'numOutputBytes': '464...",None,Apache-Spark/4.0.1 Delta-Lake/4.0.0
2,1,2025-12-07 23:19:08.204,None,None,WRITE,"{'mode': 'Append', 'partitionBy': '[]'}",None,None,None,0.0,Serializable,True,"{'numOutputRows': '32', 'numOutputBytes': '464...",None,Apache-Spark/4.0.1 Delta-Lake/4.0.0
3,0,2025-12-07 23:14:40.866,None,None,CREATE TABLE,"{'partitionBy': '[]', 'description': None, 'pr...",None,None,None,NaN,Serializable,True,{},None,Apache-Spark/4.0.1 Delta-Lake/4.0.0


Poniższe zapytanie dostarcza danych o klientach, 
którzy pojawili się jako nowi, lub zmienili swoje dane w styczniu 2021.

In [22]:
df = spark.sql("""
select id, name, address, zipcode, city, country, effectiveDate
from  (
    select id, name, address, zipcode, city, country, effectiveDate, 
           rank() over (partition by id order by to_date(effectiveDate,"dd-MM-yyyy") desc) as version
    from   source_data
    where  to_date(effectiveDate,"dd-MM-yyyy") >= date "2021-01-01"
    and    to_date(effectiveDate,"dd-MM-yyyy") < date "2021-02-01"
    )
where version = 1""")

In [23]:
df.toPandas()

,id,name,address,zipcode,city,country,effectiveDate
0,100,Vance Palmer,"P.O. Box 221, 1718 Sociis Rd.",525734,Camarones,Poland,23-01-2021
1,14,Pearl Ward,401-3122 Aliquam Av.,4449,Elbistan,Spain,02-01-2021
2,21,Desirae Morin,Ap #675-9646 Ridiculus Avenue,22382,Bergen op Zoom,New Zealand,30-01-2021
3,27,Jerome Hines,598-974 Convallis Av.,10783,Tharparkar,Colombia,15-01-2021
4,29,Darius Cole,Ap #412-3424 Eu St.,12178,Birecik,New Zealand,30-01-2021
5,4,Robin Hartman,960-7120 Lectus Rd.,833082,Fundación,Peru,26-01-2021
6,44,Kasimir Irwin,156-1322 Nulla. Road,50218,Hồ Chí Minh City,Canada,30-01-2021
7,49,Justin Burch,806-9586 Quis Rd.,83477-576,Sokoto,Sweden,06-01-2021
8,59,Coby Blackwell,311-203 Ipsum St.,249414,Belfast,Mexico,13-01-2021
9,6,Jaime Dillon,8224 Amet Road,53604,Korneuburg,Peru,17-01-2021


### Zadanie 4

Chcemy zaktualizować dane naszych klientów w oparciu o ich styczniowe wersje. Zrobimy to w dwóch krokach

1. Usuniemy z tabeli `customers` dane już nieaktualne, a następnie 
2. Wstawimy do niej dane zgodne ze styczniowymi zmianami

Usuń z tabeli `customers` tych klientów, którzy zmienili swoje dane w styczniu 2021. Identyfikacja klientów odbywać się powinna za każdym razem w oparciu o atrybut `id`.

Jeśli okaże się, że polecenie `DELETE` nie wspiera podzapytań, skorzystaj z interfejsu w Scali,
lub za pomocą poniższego kodu, w dodatkowym paragrafie uzyskaj identyfikatory klientów do usunięcia, a następnie wkomponuj uzyskaną wartość w polecenie usuwające klientów.

```python
usun_ids_df = spark.sql("""
    SELECT id AS usun
    FROM (
        SELECT
            id, name, address, zipcode, city, country, effectiveDate,
            RANK() OVER (PARTITION BY id ORDER BY to_date(effectiveDate, 'dd-MM-yyyy') DESC) AS version
        FROM source_data
        WHERE to_date(effectiveDate, 'dd-MM-yyyy') >= '2021-01-01'
            AND to_date(effectiveDate, 'dd-MM-yyyy') < '2021-02-01'
    ) tab
    WHERE version = 1
""")

# Pobranie wyników jako listy
usun_ids_list = usun_ids_df.select("usun").rdd.flatMap(lambda x: x).collect()

# Konwersja do ciągu znaków
usun_ids_str = ",".join(map(str, usun_ids_list))
```

### Rozwiązanie zadania 4 - dodatkowy paragraf

In [24]:
usun_ids_df = spark.sql("""
    SELECT id AS usun
    FROM (
        SELECT
            id, name, address, zipcode, city, country, effectiveDate,
            RANK() OVER (PARTITION BY id ORDER BY to_date(effectiveDate, 'dd-MM-yyyy') DESC) AS version
        FROM source_data
        WHERE to_date(effectiveDate, 'dd-MM-yyyy') >= '2021-01-01'
            AND to_date(effectiveDate, 'dd-MM-yyyy') < '2021-02-01'
    ) tab
    WHERE version = 1
""")

# Pobranie wyników jako listy
usun_ids_list = usun_ids_df.select("usun").rdd.flatMap(lambda x: x).collect()

# Konwersja do ciągu znaków
usun_ids_str = ",".join(map(str, usun_ids_list))

### Rozwiązanie zadania 4

In [25]:
usun_ids_df = spark.sql("""
    SELECT id AS usun
    FROM (
        SELECT
            id, name, address, zipcode, city, country, effectiveDate,
            RANK() OVER (PARTITION BY id ORDER BY to_date(effectiveDate, 'dd-MM-yyyy') DESC) AS version
        FROM source_data
        WHERE to_date(effectiveDate, 'dd-MM-yyyy') >= '2021-01-01'
            AND to_date(effectiveDate, 'dd-MM-yyyy') < '2021-02-01'
    ) tab
    WHERE version = 1
""")

usun_ids_list = usun_ids_df.select("usun").rdd.flatMap(lambda x: x).collect()
usun_ids_str = ",".join(map(str, usun_ids_list))

Za pomocą poniższego polecenia wstaw do tabeli `customers` dane klientów, 
którzy zmienili swoje dane w styczniu 2021.

In [26]:
spark.sql("""
INSERT INTO customers 
select id, name, address, zipcode, city, country, effectiveDate
from  (
    select id, name, address, zipcode, city, country, effectiveDate, 
           rank() over (partition by id order by to_date(effectiveDate,"dd-MM-yyyy") desc) as version
    from   source_data
    where  to_date(effectiveDate,"dd-MM-yyyy") >= date "2021-01-01"
    and    to_date(effectiveDate,"dd-MM-yyyy") < date "2021-02-01"
    )
where version = 1""")

DataFrame[]

Zanim przejdziemy dalej, ponownie zastanówmy się nad tym co działo się pod spodem. 

Spróbuj odpowiedzieć na dwa pytania:

1. Ile nowych plików przybyło po naszych dwóch operacjach `delete` i `insert`.
2. Ile plików będzie aktywnych - wykorzystywanych przy zapytaniach?

Znasz odpowiedzi? Sprawdźmy czy są one prawidłowe.

Wykonaj poniższe polecenia

In [27]:
%%sh
hadoop fs -ls /tmp/delta-customers

Found 7 items
drwxr-xr-x   - hadoop supergroup          0 2025-12-07 23:20 /tmp/delta-customers/_delta_log
-rw-r--r--   2 hadoop supergroup       3375 2025-12-07 23:20 /tmp/delta-customers/part-00000-2d66d538-51ad-4795-b6c8-ca2ae8279fc6-c000.snappy.parquet
-rw-r--r--   2 hadoop supergroup       4879 2025-12-07 23:19 /tmp/delta-customers/part-00000-338f7990-ae02-46dc-9a3a-c81d45d6a065-c000.snappy.parquet
-rw-r--r--   2 hadoop supergroup       1134 2025-12-07 23:15 /tmp/delta-customers/part-00000-4b5124e6-244c-4b89-bb29-7ddebc939de2-c000.snappy.parquet
-rw-r--r--   2 hadoop supergroup       4642 2025-12-07 23:19 /tmp/delta-customers/part-00000-cd2f8f28-aba4-4ca7-8a7f-47ee6a30d464-c000.snappy.parquet
-rw-r--r--   2 hadoop supergroup       4642 2025-12-07 23:19 /tmp/delta-customers/part-00000-f92fd757-ef7d-470a-9f52-39a1f796c958-c000.snappy.parquet
-rw-r--r--   2 hadoop supergroup       4879 2025-12-07 23:19 /tmp/delta-customers/part-00001-279e052d-b122-4ceb-8177-4df85df2ac54-c000.snappy.p

In [28]:
df = spark.sql("describe history customers")

In [29]:
df.toPandas()

,version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,4,2025-12-07 23:20:45.936,None,None,WRITE,"{'mode': 'Append', 'partitionBy': '[]'}",None,None,None,3.0,Serializable,True,"{'numOutputRows': '15', 'numOutputBytes': '337...",None,Apache-Spark/4.0.1 Delta-Lake/4.0.0
1,3,2025-12-07 23:19:41.970,None,None,UPDATE,"{'predicate': '[""(cast(id#1275 as int) < 50)""]'}",None,None,None,2.0,Serializable,False,"{'numDeletionVectorsUpdated': '0', 'numAddedFi...",None,Apache-Spark/4.0.1 Delta-Lake/4.0.0
2,2,2025-12-07 23:19:27.255,None,None,WRITE,"{'mode': 'Append', 'partitionBy': '[]'}",None,None,None,1.0,Serializable,True,"{'numOutputRows': '32', 'numOutputBytes': '464...",None,Apache-Spark/4.0.1 Delta-Lake/4.0.0
3,1,2025-12-07 23:19:08.204,None,None,WRITE,"{'mode': 'Append', 'partitionBy': '[]'}",None,None,None,0.0,Serializable,True,"{'numOutputRows': '32', 'numOutputBytes': '464...",None,Apache-Spark/4.0.1 Delta-Lake/4.0.0
4,0,2025-12-07 23:14:40.866,None,None,CREATE TABLE,"{'partitionBy': '[]', 'description': None, 'pr...",None,None,None,NaN,Serializable,True,{},None,Apache-Spark/4.0.1 Delta-Lake/4.0.0


Powyższa kombinacja poleceń `delete` oraz `insert` w prosty sposób 
może zostać zastąpiona poleceniem `merge`.

### Zadanie 5

Korzystając z jednego polecenia `merge` jednocześnie
*   wstaw nowych klientów, którzy pojawili się po raz pierwszy w lutym 2021
*   zaktualizuj dane adresowe oraz `effectiveDate` tych klientów, którzy w tym samym czasie te dane zmienili. 

Poniższe polecenie uzyskuje dane z obu grup klientów:

```python
select id, name, address, zipcode, city, country, effectiveDate
from  (
    select id, name, address, zipcode, city, country, effectiveDate, 
           rank() over (partition by id order by to_date(effectiveDate,"dd-MM-yyyy") desc) as version
    from   source_data
    where  to_date(effectiveDate,"dd-MM-yyyy") >= date "2021-02-01"
    and    to_date(effectiveDate,"dd-MM-yyyy") < date "2021-03-01"
    )
where version = 1
```

### Rozwiązanie zadania 5

In [30]:
spark.sql("""
MERGE INTO customers AS target
USING (
    SELECT id, name, address, zipcode, city, country, effectiveDate
    FROM (
        SELECT id, name, address, zipcode, city, country, effectiveDate, 
               RANK() OVER (PARTITION BY id ORDER BY to_date(effectiveDate,"dd-MM-yyyy") DESC) AS version
        FROM source_data
        WHERE to_date(effectiveDate,"dd-MM-yyyy") >= DATE "2021-02-01"
        AND to_date(effectiveDate,"dd-MM-yyyy") < DATE "2021-03-01"
    )
    WHERE version = 1
) AS source
ON target.id = source.id
WHEN MATCHED THEN
    UPDATE SET
        target.address = source.address,
        target.zipcode = source.zipcode,
        target.city = source.city,
        target.country = source.country,
        target.effectiveDate = source.effectiveDate
WHEN NOT MATCHED THEN
    INSERT (id, name, address, zipcode, city, country, effectiveDate)
    VALUES (source.id, source.name, source.address, source.zipcode, 
            source.city, source.country, source.effectiveDate)
""")

25/12/07 23:21:05 WARN MapPartitionsRDD: RDD 205 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting


DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

Za pomocą poniższego polecenia sprawdź, z których miesięcy pochodzą dane naszych klientów.

In [31]:
df = spark.sql("""
select substr(effectiveDate,4,10) as month,
       count(*) as how_many
from customers
group by substr(effectiveDate,4,10)
order by to_date(substr(effectiveDate,4,10),"MM-yyyy")""")

In [32]:
df.toPandas()

,month,how_many
0,10-2020,16
1,11-2020,28
2,12-2020,18
3,01-2021,15
4,02-2021,9


## Struktury plików, transakcje

Przed chwilą wykonaliśmy szereg operacji DML na naszej tabeli *customers*. 
Każda z nich, z jednej strony była oddzielną transakcją zapisaną w logach Delta Lake, z drugiej strony każda z nich dokonała pewnych zmian w plikach naszej tabeli.  

Za pomocą kilku kolejnych paragrafów rozglądniemy się na początku po strukturze plików, potem postaramy się wydobyć informacje na temat naszych transakcji.

In [33]:
# w kolumnie location znajdziesz katalog, w którym nasza tabela jest przechowywana
df = spark.sql("describe detail customers")

In [34]:
df.toPandas()

,format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures
0,delta,dad6ed48-d2bc-4f9f-bc6e-af7c2a8de703,spark_catalog.default.customers,None,hdfs://master:8020/tmp/delta-customers,2025-12-07 23:14:40.846,2025-12-07 23:21:05.673,[],[],2,8884,{},1,2,"[appendOnly, invariants]"


Sprawdź ile plików znajduje się w tym katalogu

Sprawdź ile plików ma najstarsze daty - powstało gdy po raz pierwszy załadowaliśmy do tabeli dane

W tym celu wykonaj poniższe polecenia:

In [35]:
%%sh 
hadoop fs -ls -t /tmp/delta-customers

Found 8 items
drwxr-xr-x   - hadoop supergroup          0 2025-12-07 23:21 /tmp/delta-customers/_delta_log
-rw-r--r--   2 hadoop supergroup       5509 2025-12-07 23:21 /tmp/delta-customers/part-00000-92fa3ad4-8655-425b-88e9-cd52b71d4717-c000.snappy.parquet
-rw-r--r--   2 hadoop supergroup       3375 2025-12-07 23:20 /tmp/delta-customers/part-00000-2d66d538-51ad-4795-b6c8-ca2ae8279fc6-c000.snappy.parquet
-rw-r--r--   2 hadoop supergroup       4879 2025-12-07 23:19 /tmp/delta-customers/part-00001-279e052d-b122-4ceb-8177-4df85df2ac54-c000.snappy.parquet
-rw-r--r--   2 hadoop supergroup       4879 2025-12-07 23:19 /tmp/delta-customers/part-00000-338f7990-ae02-46dc-9a3a-c81d45d6a065-c000.snappy.parquet
-rw-r--r--   2 hadoop supergroup       4642 2025-12-07 23:19 /tmp/delta-customers/part-00000-cd2f8f28-aba4-4ca7-8a7f-47ee6a30d464-c000.snappy.parquet
-rw-r--r--   2 hadoop supergroup       4642 2025-12-07 23:19 /tmp/delta-customers/part-00000-f92fd757-ef7d-470a-9f52-39a1f796c958-c000.snappy.p

In [36]:
df = spark.sql("describe history customers")

In [37]:
df.toPandas()

,version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,5,2025-12-07 23:21:05.673,None,None,MERGE,"{'matchedPredicates': '[{""actionType"":""update""...",None,None,None,4.0,Serializable,False,"{'numOutputRows': '71', 'numTargetBytesAdded':...",None,Apache-Spark/4.0.1 Delta-Lake/4.0.0
1,4,2025-12-07 23:20:45.936,None,None,WRITE,"{'mode': 'Append', 'partitionBy': '[]'}",None,None,None,3.0,Serializable,True,"{'numOutputRows': '15', 'numOutputBytes': '337...",None,Apache-Spark/4.0.1 Delta-Lake/4.0.0
2,3,2025-12-07 23:19:41.970,None,None,UPDATE,"{'predicate': '[""(cast(id#1275 as int) < 50)""]'}",None,None,None,2.0,Serializable,False,"{'numDeletionVectorsUpdated': '0', 'numAddedFi...",None,Apache-Spark/4.0.1 Delta-Lake/4.0.0
3,2,2025-12-07 23:19:27.255,None,None,WRITE,"{'mode': 'Append', 'partitionBy': '[]'}",None,None,None,1.0,Serializable,True,"{'numOutputRows': '32', 'numOutputBytes': '464...",None,Apache-Spark/4.0.1 Delta-Lake/4.0.0
4,1,2025-12-07 23:19:08.204,None,None,WRITE,"{'mode': 'Append', 'partitionBy': '[]'}",None,None,None,0.0,Serializable,True,"{'numOutputRows': '32', 'numOutputBytes': '464...",None,Apache-Spark/4.0.1 Delta-Lake/4.0.0
5,0,2025-12-07 23:14:40.866,None,None,CREATE TABLE,"{'partitionBy': '[]', 'description': None, 'pr...",None,None,None,NaN,Serializable,True,{},None,Apache-Spark/4.0.1 Delta-Lake/4.0.0


In [38]:
from delta.tables import DeltaTable

# Inicjalizacja ścieżki do tabeli Delta
delta_path = "/tmp/delta-customers"
cust_delta_table = DeltaTable.forPath(spark, delta_path)

# Pobranie pełnej historii tabeli Delta
full_hist_df = cust_delta_table.history().select(
    "version", "timestamp", "operation", "isBlindAppend", "operationMetrics"
)

In [39]:
full_hist_df.toPandas()

,version,timestamp,operation,isBlindAppend,operationMetrics
0,5,2025-12-07 23:21:05.673,MERGE,False,"{'numOutputRows': '71', 'numTargetBytesAdded':..."
1,4,2025-12-07 23:20:45.936,WRITE,True,"{'numOutputRows': '15', 'numOutputBytes': '337..."
2,3,2025-12-07 23:19:41.970,UPDATE,False,"{'numDeletionVectorsUpdated': '0', 'numAddedFi..."
3,2,2025-12-07 23:19:27.255,WRITE,True,"{'numOutputRows': '32', 'numOutputBytes': '464..."
4,1,2025-12-07 23:19:08.204,WRITE,True,"{'numOutputRows': '32', 'numOutputBytes': '464..."
5,0,2025-12-07 23:14:40.866,CREATE TABLE,True,{}


**Kilka pytań**

Zwróć uwagę, że niektóre z tych operacji mają flagę `isBlindAppend` zapaloną. **Które to były operacje?**

Każda zmiana - polecenia: `update`, `delete` czy `merge` "usuwała" dużą część (być może nawet wszystkie) aktywnych plików zastępując je nowymi. 

Co mogłoby spowodować, że liczba zastępowanych plików nie obejmowałaby wszystkich aktywnych plików?

# Podróż w czasie

Delta Lake to nie tylko wydajna obsługa (i możliwość wykonania) operacji DML, 
to także łatwość w odzyskiwaniu zniszczonych danych.

## Podstawy - czas i numery wersji

Na początku trochę podstaw. 

Sprawdź czas wykonania Twojej operacji `update`, która zmieniła kraj niektórym klientom na `Poland`.

In [40]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

# Inicjalizacja ścieżki do tabeli Delta
delta_path = "/tmp/delta-customers"
delta_table = DeltaTable.forPath(spark, delta_path)

# Pobranie historii operacji UPDATE
df = delta_table.history().where(F.col("operation") == "UPDATE").select("timestamp", "version")

In [41]:
df.toPandas()

,timestamp,version
0,2025-12-07 23:19:41.970,3


Skorzystaj teraz z tej daty, aby w poniższym paragrafie uzyskać dane jakie obowiązywały przed tą datą. 

In [43]:
df = (
    spark.read
    .format("delta")
    .option("timestampAsOf", "2025-12-07 23:19:41.970")
    .load("/tmp/delta-customers")
    .where(col("country").like("P%"))
)

In [44]:
df.toPandas()

,id,name,address,zipcode,city,country,effectiveDate
0,10,Jin Terry,467-8297 Enim,35633573,Balıkesir,Poland,06-11-2020
1,11,Isabelle Stevenson,131-4245 Eleifend. Street,16142,Hà Giang,Poland,25-11-2020
2,14,Leo Mcleod,467-8297 Enim,39153,Borås,Poland,16-10-2020
3,16,Kaitlin Landry,623-5682 Augue St.,351225,Libramont-Chevigny,Poland,29-10-2020
4,19,Alden Harper,Ap #579-2185 Sed Street,94671-72608,Châtellerault,Poland,06-12-2020
5,2,Brandon Christian,476-5064 Suspendisse Rd.,93-765,Broxburn,Poland,28-11-2020
6,20,Kathleen Pugh,7018 Cras St.,3123,Ostrowiec Świętokrzyski,Poland,16-11-2020
7,26,Ulysses Dillard,1318 Tempor Rd.,S5J 6Z2,Tuscaloosa,Poland,11-10-2020
8,28,Shaine Puckett,Ap #579-2185 Sed Street,85629,Cochrane,Poland,07-12-2020
9,33,Alexander Becker,Ap #631-7469 Curae St.,29941,Anseong,Poland,04-11-2020


In [45]:
# to samo jest możliwe za pomocą numeru wersji -- popraw go na właściwy
df = (
    spark.read
    .format("delta")
    .option("versionAsOf", "1")
    .load("/tmp/delta-customers")
    .where(col("country").like("P%"))
)

In [46]:
df.toPandas()

,id,name,address,zipcode,city,country,effectiveDate
0,20,Kathleen Pugh,7018 Cras St.,3123,Ostrowiec Świętokrzyski,Peru,16-11-2020
1,28,Shaine Puckett,Ap #579-2185 Sed Street,85629,Cochrane,Poland,07-12-2020
2,68,Clarke Carlson,2296 Vestibulum St.,163826,Logan City,Poland,28-10-2020
3,74,Vernon Casey,650-5308 Felis Rd.,42987,Mirpur,Pakistan,14-11-2020


Skorzystaj z tej daty ponownie w poniższym paragrafie, 
aby tym razem uzyskać dane jakie obowiązywały po modyfikacji.

In [47]:
df = (
    spark.read
    .format("delta")
    .option("timestampAsOf", "2025-12-07 23:19:41.970")
    .load("/tmp/delta-customers")
    .where(col("country").like("P%"))
)

In [48]:
df.toPandas()

,id,name,address,zipcode,city,country,effectiveDate
0,10,Jin Terry,467-8297 Enim,35633573,Balıkesir,Poland,06-11-2020
1,11,Isabelle Stevenson,131-4245 Eleifend. Street,16142,Hà Giang,Poland,25-11-2020
2,14,Leo Mcleod,467-8297 Enim,39153,Borås,Poland,16-10-2020
3,16,Kaitlin Landry,623-5682 Augue St.,351225,Libramont-Chevigny,Poland,29-10-2020
4,19,Alden Harper,Ap #579-2185 Sed Street,94671-72608,Châtellerault,Poland,06-12-2020
5,2,Brandon Christian,476-5064 Suspendisse Rd.,93-765,Broxburn,Poland,28-11-2020
6,20,Kathleen Pugh,7018 Cras St.,3123,Ostrowiec Świętokrzyski,Poland,16-11-2020
7,26,Ulysses Dillard,1318 Tempor Rd.,S5J 6Z2,Tuscaloosa,Poland,11-10-2020
8,28,Shaine Puckett,Ap #579-2185 Sed Street,85629,Cochrane,Poland,07-12-2020
9,33,Alexander Becker,Ap #631-7469 Curae St.,29941,Anseong,Poland,04-11-2020


In [49]:
# to samo jest możliwe za pomocą numeru wersji -- popraw go na właściwy
df = (
    spark.read
    .format("delta")
    .option("versionAsOf", "2")
    .load("/tmp/delta-customers")
    .where(col("country").like("P%"))
)

In [50]:
df.toPandas()

,id,name,address,zipcode,city,country,effectiveDate
0,20,Kathleen Pugh,7018 Cras St.,3123,Ostrowiec Świętokrzyski,Peru,16-11-2020
1,28,Shaine Puckett,Ap #579-2185 Sed Street,85629,Cochrane,Poland,07-12-2020
2,68,Clarke Carlson,2296 Vestibulum St.,163826,Logan City,Poland,28-10-2020
3,74,Vernon Casey,650-5308 Felis Rd.,42987,Mirpur,Pakistan,14-11-2020
4,20,Kathleen Pugh,7018 Cras St.,3123,Ostrowiec Świętokrzyski,Peru,16-11-2020
5,28,Shaine Puckett,Ap #579-2185 Sed Street,85629,Cochrane,Poland,07-12-2020
6,68,Clarke Carlson,2296 Vestibulum St.,163826,Logan City,Poland,28-10-2020
7,74,Vernon Casey,650-5308 Felis Rd.,42987,Mirpur,Pakistan,14-11-2020


## Rollback

### Zadanie 6

Wyobraź sobie, że ta operacja zmiany kraju na wartość `Poland` okazała się być błędna. 
Twoim zadaniem jest przywrócić wartości, które zostały nadpisane przez tą modifikację. 
Nie naprawiaj danych jeśli pojawiły się późniejsze (po Twoim poleceniu `update`) aktualizacje adresu (skorzystaj z `effectiveDate`).
Świetnie do takiej naprawy może się przydać operacja `merge`. Tym razem będzie ona miała tylko sekcję `WHEN MATCHED THEN`.

Oczywiście nie korzystaj z danych źródłowych - ich już nie ma. Jest tylko Twoja tabela Delta Lake. 

Skorzystaj z DataFrame API.

In [53]:
# błędne dane nie tylko są w historii, one są nadal w bieżącej postaci naszych danych
df = spark.sql("""
select * 
from customers 
where country like 'P%' 
  and to_date(effectiveDate,'dd-MM-yyyy') < date '2021-01-01'""")

In [54]:
df.toPandas()

,id,name,address,zipcode,city,country,effectiveDate
0,20,Kathleen Pugh,7018 Cras St.,3123,Ostrowiec Świętokrzyski,Peru,16-11-2020
1,20,Kathleen Pugh,7018 Cras St.,3123,Ostrowiec Świętokrzyski,Peru,16-11-2020
2,28,Shaine Puckett,Ap #579-2185 Sed Street,85629,Cochrane,Poland,07-12-2020
3,28,Shaine Puckett,Ap #579-2185 Sed Street,85629,Cochrane,Poland,07-12-2020
4,68,Clarke Carlson,2296 Vestibulum St.,163826,Logan City,Poland,28-10-2020
5,68,Clarke Carlson,2296 Vestibulum St.,163826,Logan City,Poland,28-10-2020
6,74,Vernon Casey,650-5308 Felis Rd.,42987,Mirpur,Pakistan,14-11-2020
7,74,Vernon Casey,650-5308 Felis Rd.,42987,Mirpur,Pakistan,14-11-2020


### Rozwiązanie zadania 6

In [55]:
from delta.tables import DeltaTable

delta_path = "/tmp/delta-customers"
delta_table = DeltaTable.forPath(spark, delta_path)

# Wczytanie poprawnych danych z wersji przed UPDATE (wersja 1)
poprawne_df = (
    spark.read.format("delta")
    .option("versionAsOf", "1")
    .load(delta_path)
)

# MERGE - przywracamy tylko te rekordy, które nie były aktualizowane później
delta_table.alias("old") \
    .merge(
        poprawne_df.alias("correct"),
        "old.id = correct.id AND old.effectiveDate = correct.effectiveDate"
    ) \
    .whenMatchedUpdate(set = {
        "country": "correct.country"
    }) \
    .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

Sprawdź czy operacja przywracania poprzedniej wersji danych się powiodła. 

In [56]:
# błędne dane nie tylko są w historii, one są nadal w bieżącej postaci naszych danych
df = spark.sql("""
select * 
from customers 
where country like 'P%' 
  and to_date(effectiveDate,'dd-MM-yyyy') < date '2021-01-01'""")

In [57]:
df.toPandas()

,id,name,address,zipcode,city,country,effectiveDate
0,20,Kathleen Pugh,7018 Cras St.,3123,Ostrowiec Świętokrzyski,Peru,16-11-2020
1,20,Kathleen Pugh,7018 Cras St.,3123,Ostrowiec Świętokrzyski,Peru,16-11-2020
2,28,Shaine Puckett,Ap #579-2185 Sed Street,85629,Cochrane,Poland,07-12-2020
3,28,Shaine Puckett,Ap #579-2185 Sed Street,85629,Cochrane,Poland,07-12-2020
4,68,Clarke Carlson,2296 Vestibulum St.,163826,Logan City,Poland,28-10-2020
5,68,Clarke Carlson,2296 Vestibulum St.,163826,Logan City,Poland,28-10-2020
6,74,Vernon Casey,650-5308 Felis Rd.,42987,Mirpur,Pakistan,14-11-2020
7,74,Vernon Casey,650-5308 Felis Rd.,42987,Mirpur,Pakistan,14-11-2020


Oczywiście należy pamiętać, że powyższa funkcjonalność to nie podróż w czasie w dowolnym zakresie. 

Nieaktualne pliki są sukcesywnie usuwane. 

Niemniej, prosta możliwość naprawy popełnionego właśnie przed chwilą błędu jest nieoceniona w świecie Big Data. To dlatego świat Big Data z zasady nie pozwala niczego modyfikować, a tworzy jedynie nowe na podstawie starego. W przypadku problemów stare, może zostać wykorzystane. 

Dokładnie to dzieje się pod maską w Delta Lake.

# Zmiany schematu

Zmiana zawartości danych to jedno. Ale świat zmienia się znacznie bardziej. Wymaga to często zmian definicji struktur naszych danych. 

Przykładowo, nasz atrybut `effectiveDate` jest ciągiem znaków, ale od dziś dobrze aby był datą. 
To samo dotyczy `id`. Od teraz wolimy, aby był liczbą. 

Tabele Delta Lake są przygotowane na takie modyfikacje. Wynika to poniekąd z formatu plików jaki jest wykorzystywany pod spodem. 

## Walidacja schematu 

Po pierwsze, Delta Lake dokonuje walidacji danych podczas wstawiania nowych danych. 

In [58]:
# przed zmianami
df = spark.sql("describe customers")

In [59]:
df.toPandas()

,col_name,data_type,comment
0,id,string,None
1,name,string,None
2,address,string,None
3,zipcode,string,None
4,city,string,None
5,country,string,None
6,effectiveDate,string,None


Spróbuj wstawić dane z marca 2021, uzupełnione o dodatkową kolumnę `endDate`.
```python
select id, 
       name, address, zipcode, city, country, 
       effectiveDate, 
       cast(null as date) as endDate
from  (
    select id, name, address, zipcode, city, country, effectiveDate,
           rank() over (partition by id order by to_date(effectiveDate,"dd-MM-yyyy") desc) as version
    from   source_data
    where  to_date(effectiveDate,"dd-MM-yyyy") >= date "2021-03-01"
    and    to_date(effectiveDate,"dd-MM-yyyy") < date "2021-04-01"
    )
where version = 1
```

In [60]:
spark.sql("""
INSERT INTO customers
select id, 
       name, address, zipcode, city, country, 
       effectiveDate, 
       cast(null as date) as endDate
from  (
    select id, name, address, zipcode, city, country, effectiveDate,
           rank() over (partition by id order by to_date(effectiveDate,"dd-MM-yyyy") desc) as version
    from   source_data
    where  to_date(effectiveDate,"dd-MM-yyyy") >= date "2021-03-01"
    and    to_date(effectiveDate,"dd-MM-yyyy") < date "2021-04-01"
    )
where version = 1""")

DataFrame[]

Dzięki parametrowi `spark.databricks.delta.schema.autoMerge.enabled=true` doszło do automatycznej integracji schematu docelowej tabeli z postacią danych, które były do niej wstawiane. 

Kolumna `endDate` jest nam potrzebna i będzie wykorzystywana w następnym zadaniu. 

W wersjach Delta Lake 2.2 i wcześniejszych należało wykorzystać poniższe rozwiązanie do uzyskania takiej integracji

```python
# Wybór danych z okresu od '2021-03-01' do '2021-04-01'
data_032021 = (
    source_data
    .filter((to_date("effectiveDate", "dd-MM-yyyy") >= '2021-03-01') & (to_date("effectiveDate", "dd-MM-yyyy") < '2021-04-01'))
    .withColumn("version", row_number().over(Window.partitionBy("id").orderBy(col("effectiveDate").desc())))
    .filter("version = 1")
    .withColumn("endDate", lit(None).cast("date"))
    .select("id", "name", "address", "zipcode", "city", "country", "effectiveDate", "endDate")
)

# Zapis danych do tabeli Delta
data_032021.write.option("mergeSchema", "true").format("delta").mode("append").saveAsTable("customers")
```


Sprawdź jak wygląda schemat Twojej tabeli po zmianach 

In [61]:
df = spark.sql("describe customers")

In [62]:
df.toPandas()

,col_name,data_type,comment
0,id,string,None
1,name,string,None
2,address,string,None
3,zipcode,string,None
4,city,string,None
5,country,string,None
6,effectiveDate,string,None
7,endDate,date,None


 
Niektóre zmiany schematu wymagają nadpisania całej zawartości tabeli 

`.mode("overwrite").option("overwriteSchema", "true")`. 

Źródłem danych może być oczywiście ta sama tabela, w tym również jej poprzednia wersja.

## Zadanie 7

Korzystając z tego mechanizmu zmień typy kolumn, o których wspominaliśmy.
Przy okazji wycofaj ostatnią aktualizację. Nie była przemyślana - dodaliśmy nowe dane (`append`) a powinniśmy uwzględnić fakt, że nie wszyscy klienci w nowym zestawie danych byli nowi, część z nich wymagała aktualizacji. W rezultacie pojawiły się duplikaty w naszych danych. 

Sprawdź, której wersji danych potrzebujesz.

In [64]:
from delta.tables import DeltaTable

# Inicjalizacja ścieżki do tabeli Delta
delta_path = "/tmp/delta-customers"
cust_delta_table = DeltaTable.forPath(spark, delta_path)

# Pobranie pełnej historii tabeli Delta
full_hist_df = cust_delta_table.history().select(
    "version", "timestamp", "operation", "isBlindAppend", "operationMetrics"
)

In [65]:
full_hist_df.toPandas()

,version,timestamp,operation,isBlindAppend,operationMetrics
0,9,2025-12-07 23:23:22.106,CREATE OR REPLACE TABLE AS SELECT,False,"{'numOutputRows': '86', 'numRemovedBytes': '12..."
1,8,2025-12-07 23:23:07.817,WRITE,True,"{'numOutputRows': '15', 'numOutputBytes': '360..."
2,7,2025-12-07 23:22:53.114,MERGE,False,"{'numOutputRows': '71', 'numTargetBytesAdded':..."
3,6,2025-12-07 23:22:32.831,MERGE,False,"{'numOutputRows': '71', 'numTargetBytesAdded':..."
4,5,2025-12-07 23:21:05.673,MERGE,False,"{'numOutputRows': '71', 'numTargetBytesAdded':..."
5,4,2025-12-07 23:20:45.936,WRITE,True,"{'numOutputRows': '15', 'numOutputBytes': '337..."
6,3,2025-12-07 23:19:41.970,UPDATE,False,"{'numDeletionVectorsUpdated': '0', 'numAddedFi..."
7,2,2025-12-07 23:19:27.255,WRITE,True,"{'numOutputRows': '32', 'numOutputBytes': '464..."
8,1,2025-12-07 23:19:08.204,WRITE,True,"{'numOutputRows': '32', 'numOutputBytes': '464..."
9,0,2025-12-07 23:14:40.866,CREATE TABLE,True,{}


### Rozwiązanie zadania 7

In [66]:
from pyspark.sql.functions import to_date, col, lit

# Wczytanie danych z wersji przed ostatnim błędnym INSERT
customers_df = (
    spark.read.format("delta")
    .option("versionAsOf", "6")  # Sprawdź w historii która wersja była przed ostatnim INSERT
    .load(delta_path)
    .withColumn("id", col("id").cast("int"))
    .withColumn("effectiveDate", to_date(col("effectiveDate"), "dd-MM-yyyy"))
    .withColumn("endDate", lit(None).cast("date"))
)

# Zapis z nadpisaniem schematu
(
    customers_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("customers")
)

25/12/07 23:23:40 WARN HiveExternalCatalog: Could not alter schema of table `default`.`customers` in a Hive compatible way. Updating Hive metastore in Spark SQL specific format.
java.lang.reflect.InvocationTargetException: null
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method) ~[?:?]
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:77) ~[?:?]
	at java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:43) ~[?:?]
	at java.base/java.lang.reflect.Method.invoke(Method.java:569) ~[?:?]
	at org.apache.spark.sql.hive.client.Shim_v4_0.alterTable(HiveShim.scala:1384) ~[spark-hive_2.13-4.0.1.jar:4.0.1]
	at org.apache.spark.sql.hive.client.HiveClientImpl.$anonfun$alterTableDataSchema$1(HiveClientImpl.scala:639) ~[spark-hive_2.13-4.0.1.jar:4.0.1]
	at scala.runtime.java8.JFunction0$mcV$sp.apply(JFunction0$mcV$sp.scala:18) ~[scala-library-2.13.16.jar:?]
	at org.apache.spark.s

In [67]:
# odczekaj chwilę aby zmiany doszły do skutku
df = spark.sql("describe customers")

In [68]:
df.toPandas()

,col_name,data_type,comment
0,id,int,None
1,name,string,None
2,address,string,None
3,zipcode,string,None
4,city,string,None
5,country,string,None
6,effectiveDate,date,None
7,endDate,date,None


Zmiany można także wymusić ręcznie. 
Dodaj jeszcze jedną kolumnę za pomocą poniższego paragrafu.

In [69]:
spark.sql("""
ALTER TABLE customers ADD COLUMNS (current boolean COMMENT 'if current data' AFTER country)
""")

DataFrame[]

In [70]:
df = spark.sql("describe customers")

In [71]:
df.toPandas()

,col_name,data_type,comment
0,id,int,None
1,name,string,None
2,address,string,None
3,zipcode,string,None
4,city,string,None
5,country,string,None
6,current,boolean,if current data
7,effectiveDate,date,None
8,endDate,date,None


# Finał 

Na zakończenie zaimplementujemy sposób utrzymywania tabel wymiarów jaki często wykorzystywany jest w hurtowniach danych.

## Slowly changing data (SCD) Type 2

Nasza tabela `customers` jest już na to gotowa. 
Kolumna `current` powinna być zapalona tylko dla najnowszych wersji danych. 
Kolumna `endDate` powinna mieć wartość zakończenia obowiązywania danej wersji danych jeśli pojawi się nowy wiersz z nową wersją danych. 



## Zadanie 8

**Zaimplementuj** funkcję `updateCustomers`, która na podstawie danych źródłowych z kolejnego *miesiąca* (parametr funkcji) będzie aktualizowała zawartość tabeli `customers` zgodnie a regułami ***SCD Type 2***.  

Po zakończonej implementacji **sprawdź jej działanie**. 

Jeśli uważasz, że obecne dane w tabeli customers powinny zostać poprawione, **dokonaj wcześniej stosownych korekt**. 

Jeśli chcesz skorzystaj ze strony:
https://docs.delta.io/latest/delta-update.html#-merge-in-scd-type-2



## Rozwiązanie zadania 8

In [80]:
spark.sql("""
UPDATE customers 
SET current = true
WHERE current IS NULL
""")
def updateCustomers(updates_df):
    from delta.tables import DeltaTable
    from pyspark.sql.functions import col, lit, current_date
    
    delta_path = "/tmp/delta-customers"
    target_table = DeltaTable.forPath(spark, delta_path)
    
    updates_df = updates_df.withColumn("current", lit(True))
    
    stagedUpdates = (
        updates_df
        .alias("updates")
        .join(target_table.toDF().alias("target"), "id")
        .where("target.current = true AND (updates.address != target.address OR updates.city != target.city OR updates.country != target.country)")
        .select(
            col("updates.id").alias("id"),
            col("updates.name").alias("name"),
            col("updates.address").alias("address"),
            col("updates.zipcode").alias("zipcode"),
            col("updates.city").alias("city"),
            col("updates.country").alias("country"),
            col("updates.effectiveDate").alias("effectiveDate"),
            col("updates.endDate").alias("endDate"),
            col("updates.current").alias("current")
        )
    )
    (
        target_table.alias("target")
        .merge(
            stagedUpdates.alias("staged_updates"),
            "target.id = staged_updates.id AND target.current = true"
        )
        .whenMatchedUpdate(
            set={
                "current": "false",
                "endDate": "staged_updates.effectiveDate"
            }
        )
        .execute()
    )
    (
        updates_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable("customers")
    )

In [ ]:
# Twoja funkcja


Masz to? 

Jeśli tak, to przyjmij gratulacje. Sprawdźmy jak to działa dla danych z marca, które wycofaliśmy. 

Na początku zobaczmy jakie to będą dane.

In [73]:
data_032021 = spark.sql("""
select id, 
       name, address, zipcode, city, country, 
       effectiveDate, 
       cast(null as date) as endDate
from  (
    select id, name, address, zipcode, city, country, effectiveDate,
           rank() over (partition by id order by to_date(effectiveDate,"dd-MM-yyyy") desc) as version
    from   source_data
    where  to_date(effectiveDate,"dd-MM-yyyy") >= date "2021-03-01"
    and    to_date(effectiveDate,"dd-MM-yyyy") < date "2021-04-01"
    )
where version = 1""")

In [74]:
data_032021.toPandas()

,id,name,address,zipcode,city,country,effectiveDate,endDate
0,12,Tucker Russo,Ap #579-2185 Sed Street,5488 CT,Mataró,United Kingdom,17-03-2021,None
1,24,Florence Landry,Ap #457-3976 Turpis. St.,725475,Pohang,Australia,27-03-2021,None
2,31,Olga Ramsey,467-8297 Enim,O1N 1T2,Bandar Lampung,United Kingdom,21-03-2021,None
3,33,Cody Alvarado,503-3360 Mattis St.,66750,Campina Grande,Turkey,22-03-2021,None
4,34,Rae Walter,656-9008 Felis. Avenue,52946,Poza Rica,India,06-03-2021,None
5,53,Kieran Preston,"337-6887 Tincidunt, St.",40825,Townsville,Nigeria,02-03-2021,None
6,55,Phoebe Craft,"337-6887 Tincidunt, St.",6338 CW,Rachecourt,Spain,20-03-2021,None
7,60,Thane Mcfarland,2050 Augue. Avenue,538383,Dublin,Belgium,27-03-2021,None
8,62,Arthur Castro,"337-6887 Tincidunt, St.",644923,Dublin,Germany,10-03-2021,None
9,66,Violet Harding,"P.O. Box 221, 1718 Sociis Rd.",3353,Nashik,Brazil,03-03-2021,None


Zobaczmy ile z tych nowych wersji klientów istnieje już w naszych danych

In [75]:
from pyspark.sql.functions import col

# Wybór identyfikatorów z ramki danych data_032021
new_ids = data_032021.select("id").collect()
new_ids_list = [str(row.id) for row in new_ids]
new_ids_str = ",".join(new_ids_list)

# Wybór danych z tabeli customers, gdzie id znajduje się w new_ids_str
df = spark.sql(f"""
    SELECT *
    FROM customers
    WHERE id IN ({new_ids_str})
""")

In [76]:
df.toPandas()

,id,name,address,zipcode,city,country,current,effectiveDate,endDate
0,33,Alexander Becker,Ap #631-7469 Curae St.,29941,Anseong,United States,True,2020-11-04,None
1,33,Alexander Becker,Ap #631-7469 Curae St.,29941,Anseong,United States,True,2020-11-04,None
2,62,Griffin Mooney,417-2786 Bibendum Ave,2441 YB,Ghizer,Costa Rica,True,2020-10-16,None
3,62,Griffin Mooney,417-2786 Bibendum Ave,2441 YB,Ghizer,Costa Rica,True,2020-10-16,None
4,69,Chaney Ray,4442 Duis Avenue,64757,Stockholm,Nigeria,True,2021-01-28,None


Uruchom swoją funkcję

In [77]:
updateCustomers(data032021)

NameError: name 'data032021' is not defined

Sprawdźmy jak wyglądają stara i nowa wersja jednego ze zaktualizowanych klientów.

In [78]:
df = spark.sql("""
select * from customers
where id = 33 """)

In [79]:
df.toPandas()

,id,name,address,zipcode,city,country,current,effectiveDate,endDate
0,33,Alexander Becker,Ap #631-7469 Curae St.,29941,Anseong,United States,True,2020-11-04,None
1,33,Alexander Becker,Ap #631-7469 Curae St.,29941,Anseong,United States,True,2020-11-04,None


Jeśli poprzednia wersja została zamknięta z odpowiednią datą a nowa z tą samą datą utworzona... 

to osiągneliśmy to o co nam chodziło... Delta Lake.